# 🗞️ News Intelligence System - Quick Start Guide

Welcome! This notebook will walk you through using the **News Intelligence Collection System** step by step.

## 📋 Table of Contents
1. [Setup & Installation](#1-setup--installation)
2. [Understanding the Project Structure](#2-understanding-the-project-structure)
3. [Loading the Pretrained Models](#3-loading-the-pretrained-models)
4. [Classifying News Articles](#4-classifying-news-articles)
5. [Understanding the Output](#5-understanding-the-output)
6. [Working with Your Own Data](#6-working-with-your-own-data)
7. [Batch Processing Large Datasets](#7-batch-processing-large-datasets)
8. [Troubleshooting Common Issues](#8-troubleshooting-common-issues)

---

## 1. Setup & Installation <a name="1-setup--installation"></a>

### Step 1.1: Install Required Dependencies

Run the cell below to install all necessary packages. This may take a few minutes.

In [ ]:
# Install core dependencies
!pip install pandas numpy scikit-learn langdetect cleantext -q

# Install web scraping & RSS
!pip install feedparser requests trafilatura beautifulsoup4 -q

# Install geospatial libraries
!pip install folium geopy geonamescache geotext -q

# Install LLM dependencies
!pip install langchain langchain-groq langchain-ollama -q

# Install utilities
!pip install python-dotenv tenacity pytz dateparser -q

print("✅ All dependencies installed successfully!")

### Step 1.2: Import Required Libraries

Now let's import all the libraries we'll need for this notebook.

In [1]:
# Core Python libraries
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Data handling
import pandas as pd
import numpy as np

# ML and text processing
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
import pickle
from langdetect import detect
import cleantext

# Import our custom classifier
from src.news_binary_classifier import NewsBinaryClassifier

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


---

## 2. Understanding the Project Structure <a name="2-understanding-the-project-structure"></a>

The project is organized as follows:

```
├── src/
│   ├── __init__.py              # Makes src a Python package
│   ├── news_binary_classifier.py # ML classifier for news articles
│   └── news_intelligence.py     # Full Streamlit application
├── models/
│   ├── Logistic Regression_62k.pkl       # Trained classification model
│   ├── word_vectorizer.pkl               # Text vectorizer (converts text → numbers)
│   └── label_encoder_transformer.pkl     # Label encoder (converts labels)
├── tests/                          # Unit tests
├── README.md                       # Project documentation
└── requirements.txt                # Python dependencies
```

### What Each Component Does:

| Component | Purpose |
|-----------|---------|
| `word_vectorizer.pkl` | Converts raw text into numerical features the ML model can understand |
| `Logistic Regression_62k.pkl` | The trained machine learning model that classifies articles |
| `label_encoder_transformer.pkl` | Converts numerical predictions back to human-readable labels |
| `news_binary_classifier.py` | The main Python class that ties everything together |

---

## 3. Loading the Pretrained Models <a name="3-loading-the-pretrained-models"></a>

### Step 3.1: Initialize the Classifier

The `NewsBinaryClassifier` class automatically loads all three model files when initialized.

In [2]:
# Path to the models directory (relative to the notebook)
models_path = 'models/'

# Initialize the classifier - this loads all models automatically
classifier = NewsBinaryClassifier(model_path=models_path)

✓ Models loaded successfully from models/


### Step 3.2: Verify Models Loaded Correctly

Let's check that all components are loaded and ready.

In [3]:
# Check model components
print("📦 Model Components Status:")
print(f"   ├── Model loaded: {classifier.model is not None}")
print(f"   ├── Vectorizer loaded: {classifier.vectorizer is not None}")
print(f"   └── Label Encoder loaded: {classifier.label_encoder is not None}")

# Check vectorizer vocabulary size
if classifier.vectorizer:
    vocab_size = len(classifier.vectorizer.vocabulary_)
    print(f"\n📊 Vectorizer Vocabulary Size: {vocab_size:,} words")

# Check label encoder classes
if classifier.label_encoder:
    classes = classifier.label_encoder.classes_
    print(f"\n🏷️  Classification Labels: {list(classes)}")

📦 Model Components Status:
   ├── Model loaded: True
   ├── Vectorizer loaded: True
   └── Label Encoder loaded: True

📊 Vectorizer Vocabulary Size: 75,472 words

🏷️  Classification Labels: ['incident', 'not-incident']


---

## 4. Classifying News Articles <a name="4-classifying-news-articles"></a>

### Step 4.1: Create Sample Data

Let's create some sample news articles to demonstrate the classifier. We have two types:
1. **Incident-related articles** - News about crimes, accidents, disasters, etc.
2. **Non-incident articles** - Regular news (sports, business, entertainment, etc.)

In [4]:
# Create sample news articles DataFrame
sample_data = {
    'title': [
        # Incident-related articles (should be classified as 'incident')
        "Breaking: Bank Robbery in Downtown Area Leaves Multiple Injured",
        "Police Investigate Fatal Car Crash on Highway 101",
        "Arson Fire Destroys Historic Building in City Center",
        "Drug Bust Leads to Multiple Arrests in Suburban Raid",
        "Protest Turns Violent as Crowds Clash with Police",
        
        # Non-incident articles (should be classified as 'non-incident')
        "Tech Giants Report Record Quarterly Earnings",
        "Local Team Wins Championship After Dramatic Final Match",
        "New Restaurant Opens Downtown with Celebrity Chef",
        "Weather Forecast: Sunny Skies Expected All Week",
        "City Council Approves New Park Development Project"
    ],
    'description': [
        "Unknown suspects fled the scene after robbing the downtown branch",
        "Three vehicles were involved in the collision, emergency services responded",
        "Fire crews battled the blaze for several hours before containing it",
        "Authorities seized drugs and weapons during the early morning operation",
        "Demonstrators gathered to protest new legislation, tensions escalated",
        "Major technology companies exceed analyst expectations with strong sales",
        "The underdog team secured victory in the final seconds of the game",
        "The new dining establishment features fusion cuisine and rooftop seating",
        "Meteorologists predict clear conditions for the next seven days",
        "The green space project will include playgrounds and walking trails"
    ],
    'article': [
        "Police are investigating a robbery at First National Bank...",
        "A serious traffic collision occurred on Highway 101...",
        "Firefighters responded to a structure fire downtown...",
        "Local police conducted a raid resulting in drug seizures...",
        "A peaceful protest escalated into violent confrontations...",
        "Apple, Microsoft, and Google all reported strong earnings...",
        "The hometown team celebrated their championship win...",
        "Chef Gordon Ramsay's new restaurant opened its doors...",
        "The national weather service predicts sunny conditions...",
        "City officials voted to approve the new park plans..."
    ]
}

# Create DataFrame
df_sample = pd.DataFrame(sample_data)
print(f"📰 Created sample dataset with {len(df_sample)} articles\n")
df_sample

📰 Created sample dataset with 10 articles



,title,description,article
0,Breaking: Bank Robbery in Downtown Area Leaves...,Unknown suspects fled the scene after robbing ...,Police are investigating a robbery at First Na...
1,Police Investigate Fatal Car Crash on Highway 101,"Three vehicles were involved in the collision,...",A serious traffic collision occurred on Highwa...
2,Arson Fire Destroys Historic Building in City ...,Fire crews battled the blaze for several hours...,Firefighters responded to a structure fire dow...
3,Drug Bust Leads to Multiple Arrests in Suburba...,Authorities seized drugs and weapons during th...,Local police conducted a raid resulting in dru...
4,Protest Turns Violent as Crowds Clash with Police,Demonstrators gathered to protest new legislat...,A peaceful protest escalated into violent conf...
5,Tech Giants Report Record Quarterly Earnings,Major technology companies exceed analyst expe...,"Apple, Microsoft, and Google all reported stro..."
6,Local Team Wins Championship After Dramatic Fi...,The underdog team secured victory in the final...,The hometown team celebrated their championshi...
7,New Restaurant Opens Downtown with Celebrity Chef,The new dining establishment features fusion c...,Chef Gordon Ramsay's new restaurant opened its...
8,Weather Forecast: Sunny Skies Expected All Week,Meteorologists predict clear conditions for th...,The national weather service predicts sunny co...
9,City Council Approves New Park Development Pro...,The green space project will include playgroun...,City officials voted to approve the new park p...


### Step 4.2: Run Classification

Now let's classify these articles. The classifier will predict whether each article is incident-related or not.

In [5]:
# Run classification on the sample data
results = classifier.predict(df_sample)

# Display results
print("🎯 Classification Results:\n")
results[['title', 'prediction']]


BINARY CLASSIFICATION - Processing 10 articles

Filtering non-English articles...
Normalizing 10 titles...
Normalizing 10 descriptions...
Normalizing 10 articles...
Vectorizing text features...
Generating predictions...

✓ Classification complete!
  Total processed: 10

Prediction Distribution:
prediction
not-incident    6
incident        4
Name: count, dtype: int64


🎯 Classification Results:



,title,prediction
0,Breaking: Bank Robbery in Downtown Area Leaves...,not-incident
1,Police Investigate Fatal Car Crash on Highway 101,incident
2,Arson Fire Destroys Historic Building in City ...,incident
3,Drug Bust Leads to Multiple Arrests in Suburba...,incident
4,Protest Turns Violent as Crowds Clash with Police,incident
5,Tech Giants Report Record Quarterly Earnings,not-incident
6,Local Team Wins Championship After Dramatic Fi...,not-incident
7,New Restaurant Opens Downtown with Celebrity Chef,not-incident
8,Weather Forecast: Sunny Skies Expected All Week,not-incident
9,City Council Approves New Park Development Pro...,not-incident


### Step 4.3: Get Predictions with Confidence Scores

For more detailed analysis, we can get probability scores showing how confident the model is in each prediction.

In [6]:
# Get predictions with probability scores
results_with_prob = classifier.predict_with_probability(df_sample)

# Display results with confidence
print("🎯 Classification Results with Confidence Scores:\n")

# Create a more readable view
display_df = results_with_prob[['title', 'prediction', 'probability']].copy()
display_df['confidence'] = display_df['probability'].apply(
    lambda x: '🔴 High' if x > 0.8 else ('🟡 Medium' if x > 0.6 else '🟢 Low')
)
display_df


CLASSIFICATION WITH PROBABILITIES - Processing 10 articles

Filtering non-English articles...
Normalizing 10 titles...
Normalizing 10 descriptions...
Normalizing 10 articles...
Vectorizing text features...
Generating predictions with probabilities...

✓ Classification complete!
  Total processed: 10

Prediction Distribution:
prediction
not-incident    6
incident        4
Name: count, dtype: int64

Confidence Statistics:
count    10.000000
mean      0.870250
std       0.177714
min       0.527128
25%       0.882663
50%       0.954831
75%       0.970014
max       0.998866
Name: probability, dtype: float64


🎯 Classification Results with Confidence Scores:



,title,prediction,probability,confidence
0,Breaking: Bank Robbery in Downtown Area Leaves...,not-incident,0.552989,🟢 Low
1,Police Investigate Fatal Car Crash on Highway 101,incident,0.527128,🟢 Low
2,Arson Fire Destroys Historic Building in City ...,incident,0.979759,🔴 High
3,Drug Bust Leads to Multiple Arrests in Suburba...,incident,0.867764,🔴 High
4,Protest Turns Violent as Crowds Clash with Police,incident,0.959673,🔴 High
5,Tech Giants Report Record Quarterly Earnings,not-incident,0.949990,🔴 High
6,Local Team Wins Championship After Dramatic Fi...,not-incident,0.998866,🔴 High
7,New Restaurant Opens Downtown with Celebrity Chef,not-incident,0.968427,🔴 High
8,Weather Forecast: Sunny Skies Expected All Week,not-incident,0.927361,🔴 High
9,City Council Approves New Park Development Pro...,not-incident,0.970543,🔴 High


---

## 5. Understanding the Output <a name="5-understanding-the-output"></a>

### What the Output Columns Mean:

| Column | Description |
|--------|-------------|
| `title` | The original article headline |
| `description` | The article description/summary |
| `article` | The full article text |
| `prediction` | The classification result (e.g., 'incident' or 'non-incident') |
| `probability` | Confidence score (0-1). Higher = more confident |
| `combined_text` | The cleaned, combined text used for classification |
| `is_english` | Boolean indicating if the text is in English |

In [7]:
# Let's look at all the columns in our results
print("📋 All columns in results DataFrame:")
print(results_with_prob.columns.tolist())

print("\n" + "="*80)
print("📊 Sample of complete output for one article:\n")

# Show full details for the first article
for col in results_with_prob.columns:
    print(f"{col}:")
    value = results_with_prob.iloc[0][col]
    if isinstance(value, str) and len(value) > 100:
        print(f"   {value[:100]}...\n")
    else:
        print(f"   {value}\n")

📋 All columns in results DataFrame:
['title', 'description', 'article', 'title_norm', 'description_norm', 'text', 'article_norm', 'prediction', 'probability']

📊 Sample of complete output for one article:

title:
   Breaking: Bank Robbery in Downtown Area Leaves Multiple Injured

description:
   Unknown suspects fled the scene after robbing the downtown branch

article:
   Police are investigating a robbery at First National Bank...

title_norm:
   breaking: bank robbery in downtown area leaves multiple injured

description_norm:
   unknown suspects fled the scene after robbing the downtown branch

text:
   breaking: bank robbery in downtown area leaves multiple injured unknown suspects fled the scene afte...

article_norm:
   police are investigating a robbery at first national bank...

prediction:
   not-incident

probability:
   0.5529889196577586



---

## 6. Working with Your Own Data <a name="6-working-with-your-own-data"></a>

### Step 6.1: Loading Your Own CSV File

Here's how to load and classify your own news data.

In [8]:
# Example: Load your own data
# Uncomment and modify the path below to use your own data

# df_your_data = pd.read_csv('path/to/your/news_data.csv')
# results = classifier.predict(df_your_data)

print("📝 To use your own data:")
print("   1. Ensure your CSV has columns: 'title', 'description', 'article'")
print("   2. Load with: pd.read_csv('your_file.csv')")
print("   3. Classify with: classifier.predict(your_dataframe)")

📝 To use your own data:
   1. Ensure your CSV has columns: 'title', 'description', 'article'
   2. Load with: pd.read_csv('your_file.csv')
   3. Classify with: classifier.predict(your_dataframe)


### Step 6.2: Using Custom Column Names

If your data has different column names, you can specify them.

In [ ]:
# Example: Your columns might be named differently
# results = classifier.predict(
#     df,
#     title_col='headline',           # if your title column is named 'headline'
#     description_col='summary',      # if your description column is named 'summary'
#     article_col='full_text'         # if your article column is named 'full_text'
# )

### Step 6.3: Headline-Only Classification (Faster)

If you only have headlines/titles and no full article text, use this method for faster processing.

In [9]:
# Headline-only classification (faster, less accurate)
df_headlines_only = df_sample[['title']].copy()

results_headlines = classifier.predict_from_headlines(df_headlines_only)

print("⚡ Headline-Only Classification Results:\n")
results_headlines[['title', 'prediction']]


HEADLINE-ONLY CLASSIFICATION - Processing 10 articles

Filtering non-English articles...
Normalizing 10 titles...
Vectorizing and predicting...

✓ Classification complete!
  Total processed: 10

Prediction Distribution:
prediction
not-incident    6
incident        4
Name: count, dtype: int64


⚡ Headline-Only Classification Results:



,title,prediction
0,Breaking: Bank Robbery in Downtown Area Leaves...,not-incident
1,Police Investigate Fatal Car Crash on Highway 101,incident
2,Arson Fire Destroys Historic Building in City ...,incident
3,Drug Bust Leads to Multiple Arrests in Suburba...,incident
4,Protest Turns Violent as Crowds Clash with Police,incident
5,Tech Giants Report Record Quarterly Earnings,not-incident
6,Local Team Wins Championship After Dramatic Fi...,not-incident
7,New Restaurant Opens Downtown with Celebrity Chef,not-incident
8,Weather Forecast: Sunny Skies Expected All Week,not-incident
9,City Council Approves New Park Development Pro...,not-incident


---

## 7. Batch Processing Large Datasets <a name="7-batch-processing-large-datasets"></a>

The classifier is optimized for processing large volumes of articles efficiently.

### Step 7.1: Filtering Results

Extract only the articles classified as incidents.

In [10]:
# Get only incident-related articles
incidents = results_with_prob[results_with_prob['prediction'] == 'incident']
non_incidents = results_with_prob[results_with_prob['prediction'] == 'non-incident']

print(f"📊 Classification Summary:")
print(f"   ├── Total Articles: {len(results_with_prob)}")
print(f"   ├── Incidents: {len(incidents)} ({len(incidents)/len(results_with_prob)*100:.1f}%)")
print(f"   └── Non-Incidents: {len(non_incidents)} ({len(non_incidents)/len(results_with_prob)*100:.1f}%)")

print("\n🚨 Incident Articles:\n")
incidents[['title', 'probability']]

📊 Classification Summary:
   ├── Total Articles: 10
   ├── Incidents: 4 (40.0%)
   └── Non-Incidents: 0 (0.0%)

🚨 Incident Articles:



,title,probability
1,Police Investigate Fatal Car Crash on Highway 101,0.527128
2,Arson Fire Destroys Historic Building in City ...,0.979759
3,Drug Bust Leads to Multiple Arrests in Suburba...,0.867764
4,Protest Turns Violent as Crowds Clash with Police,0.959673


### Step 7.2: Saving Results

Export your classification results to a file.

In [ ]:
# Save results to CSV
# results_with_prob.to_csv('classification_results.csv', index=False)

# Save only incidents to a separate file
# incidents.to_csv('incident_articles.csv', index=False)

print("💾 To save results:")
print("   results.to_csv('classification_results.csv', index=False)")
print("   incidents.to_csv('incident_articles.csv', index=False)")

---

## 8. Troubleshooting Common Issues <a name="8-troubleshooting-common-issues"></a>

### Issue 1: Model Files Not Found
```
Error: Model files not found at models/
```
**Solution:** Make sure the model files are in the correct directory:
```python
classifier = NewsBinaryClassifier(model_path='path/to/models/')
```

### Issue 2: Missing Columns
```
Error: DataFrame must contain 'title' column
```
**Solution:** Ensure your DataFrame has the required columns, or specify custom column names:
```python
results = classifier.predict(df, title_col='your_title_col')
```

### Issue 3: Non-English Text
The classifier is trained primarily on English text. Non-English articles may have lower accuracy.

### Issue 4: Empty or Null Values
The classifier handles missing values, but for best results, ensure:
- Title column is never empty
- Fill missing descriptions/articles with the title text

---

## 🎓 Quick Reference

### Basic Usage
```python
from src.news_binary_classifier import NewsBinaryClassifier
import pandas as pd

# Initialize
classifier = NewsBinaryClassifier(model_path='models/')

# Load your data
df = pd.read_csv('your_news_data.csv')

# Classify
results = classifier.predict(df)

# Get confidence scores
results = classifier.predict_with_probability(df)

# Filter incidents
incidents = results[results['prediction'] == 'incident']
```

### Methods Available
| Method | Description | Speed |
|--------|-------------|-------|
| `predict(df)` | Classify with title + description + article | Medium |
| `predict_from_headlines(df)` | Classify with title only | Fast |
| `predict_with_probability(df)` | Classify with confidence scores | Medium |

---

**Happy Classifying! 🎉**